# FFAI RAG Pipeline Demo

This notebook demonstrates the full Retrieval-Augmented Generation (RAG) pipeline in FFAI.
All RAG/VectorStore interactions are mocked so the notebook runs without chromadb.

## Section 1: Setup

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

print(f'Project root: {_project_root}')

In [ ]:
from examples._rag_data.download import download

docs = download()
for name, path in docs.items():
    print(f'  {name}: {path.stat().st_size:,} bytes')

In [ ]:
python_tutorial = docs['python_tutorial.md'].read_text(encoding='utf-8')
api_docs = docs['api_docs.md'].read_text(encoding='utf-8')

print(f'python_tutorial.md: {len(python_tutorial):,} chars')
print(f'api_docs.md:        {len(api_docs):,} chars')

<div class="page-break"></div>

---

## Section 2: The RAG Pipeline Overview

The FFAI RAG pipeline follows this flow:

```
Document → Chunk → Embed → Store → Query → Retrieve → Format
```

| Step | Component | Description |
|------|-----------|-------------|
| 1 | Chunking | Split documents into manageable pieces |
| 2 | Embedding | Convert chunks to vector representations |
| 3 | BM25 Indexing | Sparse keyword-based retrieval |
| 4 | Hierarchical Index | Parent-child chunk relationships |
| 5 | Contextual Embeddings | Add document context to chunk embeddings |
| 6 | RAG | High-level unified API (mocked) |
| 7 | Integration | Inject retrieved context into LLM prompts |

In [ ]:
from src.rag import Embeddings, TextChunk, get_chunker
from src.rag.indexing import BM25Index, ContextualEmbeddings, HierarchicalIndex

print('Imports successful')
print(f'Available chunkers: {get_chunker.__module__}')
print(f'TextChunk fields: {[f.name for f in TextChunk.__dataclass_fields__.values()]}')

<div class="page-break"></div>

---

## Section 3: Step 1 - Chunking Documents

In [ ]:
md_chunker = get_chunker('markdown', chunk_size=500)
tutorial_chunks = md_chunker.chunk(python_tutorial)

char_chunker = get_chunker('character', chunk_size=800)
api_chunks = char_chunker.chunk(api_docs)

print(f'Tutorial chunks (markdown, size=500):  {len(tutorial_chunks)}')
print(f'API docs chunks (character, size=800): {len(api_chunks)}')

In [ ]:
print('=== Tutorial: first 2 chunks ===')
for i, chunk in enumerate(tutorial_chunks[:2]):
    preview = chunk.content[:120].replace(chr(10), ' ')
    print(f'  Chunk {i}: chars={chunk.start_char}-{chunk.end_char} | {preview}...')

print()
print('=== API Docs: first 2 chunks ===')
for i, chunk in enumerate(api_chunks[:2]):
    preview = chunk.content[:120].replace(chr(10), ' ')
    print(f'  Chunk {i}: chars={chunk.start_char}-{chunk.end_char} | {preview}...')

<div class="page-break"></div>

---

## Section 4: Step 2 - Embedding Chunks (Mocked)

We mock `litellm.embedding` to return fake 1024-dimensional vectors so no API key is needed.

In [ ]:
import random
import unittest.mock

random.seed(42)

_fake_dim = 1024

class _FakeEmbeddingData(dict):
    def __init__(self, idx, dim):
        super().__init__(embedding=[random.gauss(0, 1) / 10 for _ in range(dim)], index=idx)
        self.embedding = self['embedding']
        self.index = self['index']

async def _fake_litellm_embedding(model, input, **kwargs):
    texts = input if isinstance(input, list) else [input]
    data = [_FakeEmbeddingData(i, _fake_dim) for i in range(len(texts))]
    return type('R', (), {'data': data})()

_patcher = unittest.mock.patch('litellm.aembedding', side_effect=_fake_litellm_embedding)
_patcher.start()

embeddings = Embeddings(model='mistral/mistral-embed', api_key='fake-key', cache_enabled=False)

sample_texts = [tutorial_chunks[i].content for i in range(3)]
vectors = embeddings.embed(sample_texts)

print(f'Embedded {len(vectors)} chunks')
print(f'Dimensions per vector: {len(vectors[0])}')
for i, vec in enumerate(vectors):
    vals = ', '.join(f'{v:.4f}' for v in vec[:5])
    print(f'  Chunk {i}: first 5 values = [{vals}]')

<div class="page-break"></div>

---

## Section 5: Step 3 - BM25 Indexing

BM25 provides sparse keyword-based retrieval. No external dependencies needed.

In [ ]:
bm25 = BM25Index()

for i, chunk in enumerate(tutorial_chunks):
    bm25.add_document(
        doc_id=f'tutorial_{i}',
        content=chunk.content,
        metadata=chunk.metadata,
    )

print(f'Indexed {bm25.count()} documents in BM25')

In [ ]:
results = bm25.search('async programming', n_results=3)

print('BM25 search: "async programming"')
print(f'Top {len(results)} results:\n')
for i, r in enumerate(results):
    preview = r['content'][:100].replace(chr(10), ' ')
    score = r['score']
    print(f'  [{i+1}] score={score:.4f} | {preview}...')

<div class="page-break"></div>

---

## Section 6: Step 4 - Hierarchical Index

The `HierarchicalIndex` stores parent-child chunk relationships, allowing retrieval
to return fine-grained child chunks with parent context for richer LLM prompts.

In [ ]:
hier = HierarchicalIndex(include_parent_context=True)

hier.add_chunk('parent_1', 'Introduction to async programming in Python...', hierarchy_level=0)
hier.add_chunk('child_1a', 'A coroutine is a function defined with async def.', parent_id='parent_1', hierarchy_level=1)
hier.add_chunk('child_1b', 'The await keyword pauses the coroutine until completion.', parent_id='parent_1', hierarchy_level=1)

hier.add_chunk('parent_2', 'Error handling and timeouts in async code...', hierarchy_level=0)
hier.add_chunk('child_2a', 'Use asyncio.wait_for() to impose a timeout.', parent_id='parent_2', hierarchy_level=1)

print(f'Parents: {hier.count_parents()}, Children: {hier.count_children()}, Total: {hier.count()}')

In [ ]:
mock_results = [{'id': 'child_1a', 'content': 'A coroutine is a function defined with async def.', 'score': 0.92}]

enhanced = hier.enhance_results_with_context(mock_results)

print('Before enhance:', list(mock_results[0].keys()))
print('After enhance: ', list(enhanced[0].keys()))
print(f"\nParent content added: '{enhanced[0]['parent_content']}'")

<div class="page-break"></div>

---

## Section 7: Step 5 - Contextual Embeddings

`ContextualEmbeddings` prepends document context (title, section) to chunk content
before embedding, improving semantic understanding.

In [ ]:
ctx = ContextualEmbeddings()

chunks_with_meta = [
    {'content': 'A coroutine is a function defined with async def.', 'metadata': {'header': 'Basic Coroutines'}},
    {'content': 'Use asyncio.gather() for concurrent I/O operations.', 'metadata': {'header': 'Best Practices'}},
]

prepared = ctx.prepare_chunks_batch(chunks_with_meta, document_title='python_tutorial.md')

for i, (original, context_prepared) in enumerate(zip(chunks_with_meta, prepared)):
    print(f'--- Chunk {i} ---')
    print(f'BEFORE: {original["content"]}')
    print(f'AFTER:  {context_prepared}')
    print()

<div class="page-break"></div>

---

## Section 8: Full Pipeline with RAG (Mocked)

RAG requires chromadb. We create a mock that demonstrates the same API surface:
- `add_document()` - index a document
- `search()` - find relevant chunks
- `format_results_for_prompt()` - format for LLM injection
- `search_and_format()` - convenience method

In [ ]:
class MockRAG:
    """Mock RAG demonstrating the API without chromadb."""

    def __init__(self, **kwargs):
        self._bm25 = BM25Index()
        self._doc_counter = 0

    def add_document(self, content, metadata=None, reference_name=None, **kw):
        chunker = get_chunker('recursive', chunk_size=500)
        chunks = chunker.chunk(content, metadata=metadata or {})
        for i, chunk in enumerate(chunks):
            doc_id = f'{reference_name or "doc"}_{i}'
            self._bm25.add_document(doc_id, chunk.content, chunk.metadata)
        self._doc_counter += len(chunks)
        return len(chunks)

    def search(self, query, n_results=5, **kw):
        return self._bm25.search(query, n_results=n_results)

    def format_results_for_prompt(self, results, max_chars=None):
        if not results:
            return ''
        formatted = []
        total = 0
        for i, r in enumerate(results, 1):
            content = r.get('content', '')
            source = r.get('metadata', {}).get('reference_name', 'unknown')
            score = r.get('score', 0.0)
            chunk_text = f'[{i}] (source: {source}, relevance: {score:.2f})\n{content}\n'
            if max_chars and total + len(chunk_text) > max_chars:
                break
            formatted.append(chunk_text)
            total += len(chunk_text)
        return ''.join(formatted)

    def search_and_format(self, query, n_results=5, max_chars=None, **kw):
        results = self.search(query, n_results=n_results)
        return self.format_results_for_prompt(results, max_chars=max_chars)

rag = MockRAG()
n = rag.add_document(python_tutorial, metadata={'reference_name': 'python_tutorial.md'})
print(f'Indexed {n} chunks from python_tutorial.md')

In [ ]:
formatted = rag.search_and_format('async programming', n_results=3)

print('Formatted output for LLM prompt injection:')
print('=' * 60)
print(formatted)
print('=' * 60)
print(f'Total chars: {len(formatted)}')

<div class="page-break"></div>

---

## Section 9: Integration with FFAI

Shows how RAG results integrate with FFAI's `generate_response()`. This is conceptual - no real LLM calls.

In [ ]:
RAG_PROMPT_TEMPLATE = """You are a helpful assistant. Use the following context to answer the question.

Context:
{context}

Question: {query}

Answer:"""

query = 'How do I run multiple async tasks concurrently?'
context = rag.search_and_format(query, n_results=3)
final_prompt = RAG_PROMPT_TEMPLATE.format(context=context, query=query)

print(f'Prompt length: {len(final_prompt):,} chars')
nl = chr(10)
print(f'Context chunks: {context.count(nl + "[")}')
print()
print('--- Final prompt (truncated) ---')
print(final_prompt[:500] + '...')

<div class="page-break"></div>

---

## Section 10: Summary

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {'Step': 1, 'Component': 'Chunking', 'Module': 'src.rag.splitters', 'Key Method': 'get_chunker().chunk()'},
    {'Step': 2, 'Component': 'Embedding', 'Module': 'src.rag.embed', 'Key Method': 'Embeddings.embed()'},
    {'Step': 3, 'Component': 'BM25 Index', 'Module': 'src.rag.indexing.bm25', 'Key Method': 'BM25Index.search()'},
    {'Step': 4, 'Component': 'Hierarchical Index', 'Module': 'src.rag.indexing.hierarchical', 'Key Method': 'enhance_results_with_context()'},
    {'Step': 5, 'Component': 'Contextual Embeddings', 'Module': 'src.rag.indexing.contextual', 'Key Method': 'prepare_chunks_batch()'},
    {'Step': 6, 'Component': 'RAG', 'Module': 'src.rag.client', 'Key Method': 'search_and_format()'},
    {'Step': 7, 'Component': 'Integration', 'Module': 'Prompt template', 'Key Method': 'RAG_PROMPT_TEMPLATE.format()'},
])

print(summary.to_string(index=False))